# KOZ-3 · build v23b = v22 + DINOv3-SAT hillfort localiser (platform-safe)

v23 scored exactly v22 because the localiser needed PIL / torchvision (absent on the platform) and silently fell back to the scene centre. v23b reads the normalisation from JSON, and the parity check now runs with PIL / torchvision blocked and the archive vendor first.

v23 is the **exact v22 archive** with one change: satellite small scenes that pass v22's own hillfort gate (p ≥ 0.1) emit
the hillfort at the DINOv3 probe-map peak instead of the scene centre (nested OOF satellite 0.3797 → 0.4097, no fold worse,
k* = 11, P = 1.00). Everything else in v22 stays byte-identical.

Steps: locate the v22 zip → build (fp16 if it fits under 2 GB with margin, else int8) with a parity gate on 12 training
scenes → paired rehearsal v22 vs v23 on the local rehearsal set (runtime + score) → summary. ~20–40 min.

In [ ]:
import os, glob, subprocess, json
MAIN = "/home/nathan/Expedition Competition"; WT = os.path.join(MAIN, "Codex Work", "dinov3_wt"); os.chdir(WT)
subprocess.run(["git", "fetch", "origin", "feat/physics-gates"], check=True)
subprocess.run(["git", "checkout", "origin/feat/physics-gates", "--", "koz3/hf_dino_loc.py", "scripts/make_v23_from_v22.py",
                "models/hf_dino_loc.npz"], check=True)
print("hf_dino_loc without AutoImageProcessor:", "AutoImageProcessor.from_pretrained(backbone" not in open("koz3/hf_dino_loc.py").read())
cands = sorted(glob.glob(os.path.join(MAIN, "**", "*v22*.zip"), recursive=True), key=os.path.getmtime)
for c in cands: print(f"{os.path.getsize(c) / 1e9:6.3f} GB  {c}")
V22 = [c for c in cands if "etgr_fusion_final" in c][-1] if any("etgr_fusion_final" in c for c in cands) else cands[-1]
print("using v22:", V22)
for p in ["work/hf_dinov3/tokens.npz", "work/hf_dinov3/tokens768.npz"]: print(("OK      " if os.path.exists(p) else "MISSING ") + p)

## Build (parity gate: the packed localiser must reproduce the validated points on 12 training scenes)

In [ ]:
size = os.path.getsize(V22) / 1e9
use_int8 = size + 0.56 > 1.90            # keep ≥ 0.1 GB margin under the 2 GB limit
V23 = os.path.join(os.path.dirname(V22), "koz3_submission_v23b_dino_hillfort_" + ("int8" if use_int8 else "fp16") + ".zip")
print(f"v22 {size:.3f} GB → {'int8 (+0.21 GB)' if use_int8 else 'fp16 (+0.56 GB)'}  →  {V23}")
args = ["python", "scripts/make_v23_from_v22.py", V22, V23, "--n", "12"] + (["--int8"] if use_int8 else [])
r = subprocess.run(args, capture_output=True, text=True, env={**os.environ, "HF_HUB_OFFLINE": "1"})
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "warn" not in l.lower() and "loading" not in l.lower() and "shards" not in l.lower())[-4000:])
print("BUILD", "OK" if r.returncode == 0 and os.path.exists(V23) else "FAILED")

## Paired rehearsal v22 vs v23 (same local input, platform-style run)
Finds a local rehearsal set (input folder + GT geojson). The satellite hillfort lines in the v23 log should read
`DINOv3 hillfort point (map max …)`.

In [ ]:
sets = []
for base in (MAIN, WT):
    for d in sorted(glob.glob(os.path.join(base, "work", "rehearsal*"))):
        gts = sorted(glob.glob(os.path.join(d, "gt*.geojson"))); inp = os.path.join(d, "input")
        if gts and os.path.isdir(inp): sets.append((inp, gts[-1]))
for s in sets: print(s)
if sets:
    inp, gt = sets[0]
    for zp in (V22, V23):
        print("=" * 20, os.path.basename(zp))
        r = subprocess.run(["python", "scripts/rehearse.py", zp, inp, gt], capture_output=True, text=True)
        out = r.stdout + r.stderr
        print("\n".join(l for l in out.splitlines() if any(k in l for k in ("score", "Score", "final", "sec", "min", "DINOv3", "hillfort", "error", "Error", "Traceback")))[-3000:])
else:
    print("no local rehearsal set found — ask Codex to run scripts/rehearse.py on", V23)

## Summary to paste back

In [ ]:
print("v22:", V22, f"{os.path.getsize(V22)/1e9:.3f} GB")
print("v23:", V23, f"{os.path.getsize(V23)/1e9:.3f} GB" if os.path.exists(V23) else "missing")